In [17]:
import pandas as pd

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

print("Ukuran dataset:", df.shape)

print("\nProporsi kelas Churn:")
print(df["Churn"].value_counts(normalize=True))

print("\nTipe data:")
print(df.dtypes)

Ukuran dataset: (7043, 21)

Proporsi kelas Churn:
Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64

Tipe data:
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object


In [18]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

df = df.dropna(subset=["TotalCharges"])

In [19]:
df["Churn"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

In [20]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

In [21]:
X = pd.get_dummies(X, drop_first=True)

In [22]:
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [23]:
print("Data training:", X_tr.shape)
print("Data testing:", X_te.shape)

print("\nProporsi churn training:")
print(y_tr.value_counts(normalize=True))

print("\nProporsi churn testing:")
print(y_te.value_counts(normalize=True))

Data training: (5625, 7061)
Data testing: (1407, 7061)

Proporsi churn training:
Churn
0    0.734222
1    0.265778
Name: proportion, dtype: float64

Proporsi churn testing:
Churn
0    0.734186
1    0.265814
Name: proportion, dtype: float64


In [24]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42
)

rf.fit(X_tr, y_tr)

RandomForestClassifier(class_weight='balanced', n_estimators=300,
                       random_state=42)

In [25]:
from sklearn.metrics import classification_report, roc_auc_score

y_pred = rf.predict(X_te)
y_proba = rf.predict_proba(X_te)[:, 1]

print("Classification Report:")
print(classification_report(y_te, y_pred))

print("ROC-AUC:", roc_auc_score(y_te, y_proba))

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1033
           1       0.63      0.51      0.56       374

    accuracy                           0.79      1407
   macro avg       0.73      0.70      0.71      1407
weighted avg       0.78      0.79      0.78      1407

ROC-AUC: 0.8309050012683064


In [27]:
hasil_prediksi = pd.DataFrame({
    "Actual_Churn": y_te.values,
    "Predicted_Churn": y_pred,
    "Churn_Probability": y_proba
})

print(hasil_prediksi.head(10))

   Actual_Churn  Predicted_Churn  Churn_Probability
0             0                0           0.020000
1             0                1           0.733333
2             0                0           0.010000
3             1                0           0.093333
4             0                0           0.163333
5             1                0           0.356667
6             0                0           0.013333
7             0                0           0.140000
8             1                1           0.740000
9             0                0           0.006667


Kesimpulan:

Model Random Forest digunakan untuk memprediksi kemungkinan pelanggan melakukan churn dengan mempertimbangkan ketidakseimbangan kelas melalui class_weight="balanced". 
Berdasarkan hasil evaluasi, model menghasilkan nilai precision, recall, F1-score untuk kelas churn (kelas 1), serta ROC-AUC sebesar 0.8309050012683064. 
Nilai recall menunjukkan kemampuan model dalam mendeteksi pelanggan yang benar-benar melakukan churn, sedangkan ROC-AUC menunjukkan kemampuan model dalam membedakan pelanggan churn dan tidak churn. 
Berdasarkan probabilitas prediksi, pelanggan dengan nilai probabilitas churn yang tinggi dapat menjadi prioritas untuk diberikan strategi retensi.